In [1]:
import numpy as np
import pandas as pd
from torch.utils.data import Dataset
import albumentations as A
import pytorch_lightning as pl
import torch
from torch.optim import Adam
from tqdm.auto import tqdm
import evaluate

from pytorch_lightning.callbacks.early_stopping import EarlyStopping
from pytorch_lightning.callbacks.model_checkpoint import ModelCheckpoint
from transformers import MaskFormerImageProcessor
from transformers import MaskFormerForInstanceSegmentation
from torch.utils.data import DataLoader

from src.utils.dataset import load_foodseg103_splits
from src.constants.category_id import CATEGORY_ID
from src.utils.visualization import predict_random_images

c:\Users\ferna\AppData\Local\Programs\Python\Python312\Lib\site-packages\albumentations\__init__.py:28: UserWarning: A new version of Albumentations is available: '2.0.7' (you have '2.0.5'). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


In [2]:
SEED = 42
IMAGE_SIZE = 512
LEARNING_RATE = 5e-5
BATCH_SIZE = 2
NUM_EPOCHS = 1
MODEL_NAME = "facebook/maskformer-swin-base-ade"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
class SemanticSegmentationFoodDataset(Dataset):
    def __init__(self, dataset:pd.DataFrame, transform:A.Compose=None):
        self.dataset = dataset
        self.transform = transform

    def __len__(self):
        return self.dataset.shape[0]
    
    def __getitem__(self, idx):
        original_image = np.array(self.dataset[idx]['image'])
        original_segmentation_map = np.array(self.dataset[idx]['label'])
        transformed = self.transform(image=original_image, mask=original_segmentation_map)
        image, segmentation_map = transformed['image'], transformed['mask']
        return image, segmentation_map, original_image, original_segmentation_map

In [4]:
ADE_MEAN = np.array([123.675, 116.280, 103.530]) / 255
ADE_STD = np.array([58.395, 57.120, 57.375]) / 255

train_transform = A.Compose([
    A.Resize(512, 512),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),  # Adiciona alguma variabilidade extra
    A.RandomBrightnessContrast(p=0.4),
    A.ColorJitter(p=0.3),  # Mudança de cor, saturação, etc
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=20, p=0.5),

    A.GaussianBlur(blur_limit=(3,5), p=0.1),  # Um pouco de desfoque
    A.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
    A.pytorch.ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(512, 512),
    A.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
    A.pytorch.ToTensorV2()
])

c:\Users\ferna\AppData\Local\Programs\Python\Python312\Lib\site-packages\albumentations\core\validation.py:87: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


In [5]:
df = load_foodseg103_splits()

In [6]:
train_dataset = SemanticSegmentationFoodDataset(df["train"], train_transform)
test_dataset = SemanticSegmentationFoodDataset(df["test"], val_transform)
val_dataset = SemanticSegmentationFoodDataset(df["validation"], val_transform)

In [7]:
preprocessor = MaskFormerImageProcessor(ignore_index=0, reduce_labels=False, do_resize=False, do_rescale=False, do_normalize=False)

In [8]:
def collate_fn(batch):
    inputs = list(zip(*batch))
    images = inputs[0]
    segmentation_maps = inputs[1]
    batch = preprocessor(
        images,
        segmentation_maps=segmentation_maps,
        return_tensors="pt",
    )
    batch["original_images"] = inputs[2]
    batch["original_segmentation_maps"] = inputs[3]
    return batch

train_dataloader = DataLoader(train_dataset, batch_size=2, shuffle=True, collate_fn=collate_fn)
val_dataloader = DataLoader(val_dataset, batch_size=2, shuffle=False, collate_fn=collate_fn)

model = MaskFormerForInstanceSegmentation.from_pretrained(
    MODEL_NAME,
    id2label=CATEGORY_ID,
    ignore_mismatched_sizes=True
)
layers = model.model.pixel_level_module.encoder.model.encoder.layers
for i, layer in enumerate(layers):
    if i < 2:
        for param in layer.parameters():
            param.requires_grad = False

total_params = sum(p.numel() for p in model.parameters()) / 1e6
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
frozen_params = total_params - trainable_params

print(f"Total de parâmetros: {total_params:.2f}M")
print(f"Parâmetros treináveis: {trainable_params:.2f}M")
print(f"Parâmetros congelados: {frozen_params:.2f}M")

In [9]:
class SegmentationLightningModule(pl.LightningModule):
    def __init__(self, train_dataloader, val_dataloader, preprocessor, lr=5e-5):
        super().__init__()
        self.id2label = CATEGORY_ID
        self.model = MaskFormerForInstanceSegmentation.from_pretrained(
            MODEL_NAME,
            id2label=self.id2label,
            ignore_mismatched_sizes=True
        )

        self.val_mean_iou = evaluate.load("mean_iou")
        self.validation_step_outputs = []

        self.train_mean_iou = evaluate.load("mean_iou")

        self.preprocessor = preprocessor
        self.lr = lr
        self.train_dl = train_dataloader
        self.val_dl = val_dataloader

    def train_dataloader(self):
        return self.train_dl
    
    def val_dataloader(self):
        return self.val_dl

    def configure_optimizers(self):
        return Adam(self.model.parameters(), lr=self.lr)

    def forward(self, pixel_values, mask_labels=None, class_labels=None):
        return self.model(
            pixel_values=pixel_values,
            mask_labels=mask_labels,
            class_labels=class_labels
        )

    def training_step(self, batch, batch_idx):
        device = batch["pixel_values"].device
        outputs = self(
            pixel_values=batch["pixel_values"],
            mask_labels=[labels.to(device) for labels in batch["mask_labels"]],
            class_labels=[labels.to(device) for labels in batch["class_labels"]],
        )
        loss = outputs.loss
        self.log("train_loss", loss, prog_bar=True, on_step=True, on_epoch=True)

        target_sizes = [(img.shape[0], img.shape[1]) for img in batch["original_images"]]
        predicted_maps = self.preprocessor.post_process_semantic_segmentation(outputs, target_sizes=target_sizes)
        ground_truth_maps = batch["original_segmentation_maps"]
        self.train_mean_iou.add_batch(references=ground_truth_maps, predictions=predicted_maps)
        return loss

    def validation_step(self, batch, batch_idx):
        device = batch["pixel_values"].device
        with torch.no_grad():
            outputs = self(pixel_values=batch["pixel_values"].to(device))
        if outputs.loss is not None:
            self.validation_step_outputs.append(outputs.loss)
        target_sizes = [(img.shape[0], img.shape[1]) for img in batch["original_images"]]
        predicted_maps = self.preprocessor.post_process_semantic_segmentation(outputs, target_sizes=target_sizes)
        ground_truth_maps = batch["original_segmentation_maps"]
        self.val_mean_iou.add_batch(references=ground_truth_maps, predictions=predicted_maps)

    def on_validation_epoch_end(self):
        if len(self.validation_step_outputs) != 0:
            avg_val_loss = torch.stack(self.validation_step_outputs).mean()
            self.log("val_loss", avg_val_loss, prog_bar=True)
            self.validation_step_outputs.clear() #free memory
        mean_iou = self.val_mean_iou.compute(num_labels=len(self.id2label), ignore_index=0)["mean_iou"]
        self.log("val_mean_iou", mean_iou, prog_bar=True)

    def on_train_epoch_end(self):
        mean_iou = self.train_mean_iou.compute(num_labels=len(self.id2label), ignore_index=0)["mean_iou"]
        self.log("train_mean_iou", mean_iou, prog_bar=True)

In [10]:
segmentation_lightning_module = SegmentationLightningModule(train_dataloader, val_dataloader, preprocessor)

Some weights of MaskFormerForInstanceSegmentation were not initialized from the model checkpoint at facebook/maskformer-swin-base-ade and are newly initialized because the shapes did not match:
- class_predictor.weight: found shape torch.Size([151, 256]) in the checkpoint and torch.Size([105, 256]) in the model instantiated
- class_predictor.bias: found shape torch.Size([151]) in the checkpoint and torch.Size([105]) in the model instantiated
- criterion.empty_weight: found shape torch.Size([151]) in the checkpoint and torch.Size([105]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [11]:
early_stop_callback = EarlyStopping(
    monitor="val_mean_iou",
    min_delta=0,
    patience=3, 
    mode="max"
)

model_name_formatted = MODEL_NAME.replace('/', '-')
image_size_formatted = f"{IMAGE_SIZE}x{IMAGE_SIZE}"

checkpoint_callback = ModelCheckpoint(
    dirpath="./src/model",
    filename=f"{model_name_formatted}-{image_size_formatted}-{{epoch}}-{{val_mean_iou:.4f}}",
    save_top_k=1,
    monitor="val_mean_iou",
    mode="max"
)

pl.seed_everything(SEED, workers=True)
trainer = pl.Trainer(
    callbacks=[checkpoint_callback],
    max_epochs=NUM_EPOCHS
)

Seed set to 42
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [12]:
trainer.fit(segmentation_lightning_module)

c:\Users\ferna\AppData\Local\Programs\Python\Python312\Lib\site-packages\pytorch_lightning\callbacks\model_checkpoint.py:654: Checkpoint directory C:\Users\ferna\Documents\food-macro-analyzer\src\model exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name  | Type                              | Params | Mode
-------------------------------------------------------------------
0 | model | MaskFormerForInstanceSegmentation | 101 M  | eval
-------------------------------------------------------------------
101 M     Trainable params
0         Non-trainable params
101 M     Total params
407.278   Total estimated model params size (MB)
0         Modules in train mode
646       Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

c:\Users\ferna\AppData\Local\Programs\Python\Python312\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
c:\Users\ferna\AppData\Local\Programs\Python\Python312\Lib\site-packages\datasets\features\image.py:347: UserWarning: Downcasting array dtype int64 to int32 to be compatible with 'Pillow'
  warnings.warn(f"Downcasting array dtype {dtype} to {dest_dtype} to be compatible with 'Pillow'")
C:\Users\ferna\.cache\huggingface\modules\evaluate_modules\metrics\evaluate-metric--mean_iou\9e450724f21f05592bfb0255fe2fa576df8171fa060d11121d8aecfff0db80d0\mean_iou.py:259: RuntimeWarning: invalid value encountered in divide
  iou = total_area_intersect / total_area_union
C:\Users\ferna\.cache\huggingface\modules\evaluate_modules\metrics\evaluate-metric--mean_iou\9e450724f21f

Training: |          | 0/? [00:00<?, ?it/s]

c:\Users\ferna\AppData\Local\Programs\Python\Python312\Lib\site-packages\pytorch_lightning\utilities\data.py:79: Trying to infer the `batch_size` from an ambiguous collection. The batch size we found is 2. To avoid any miscalculations, use `self.log(..., batch_size=batch_size)`.


OutOfMemoryError: CUDA out of memory. Tried to allocate 8.15 GiB. GPU 0 has a total capacity of 6.00 GiB of which 0 bytes is free. Of the allocated memory 4.89 GiB is allocated by PyTorch, and 177.02 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
#predict image
model = segmentation_lightning_module.model
model.eval()
model.to(DEVICE)

In [ ]:
# Predict on a random image
for i, batch in enumerate(val_dataloader):
    with torch.no_grad():
        pixel_values = batch["pixel_values"].to(DEVICE)
        outputs = model(pixel_values=pixel_values)
        target_sizes = [(img.shape[0], img.shape[1]) for img in batch["original_images"]]
        predicted_maps = preprocessor.post_process_semantic_segmentation(outputs, target_sizes=target_sizes)
        ground_truth_maps = batch["original_segmentation_maps"]
        original_images = batch["original_images"]
        break

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.imshow(original_images[0])

In [ ]:
plt.imshow(predicted_maps[0].cpu())